# Sentiment Trading Signals — End-to-End Pipeline Demo

This notebook demonstrates the full pipeline from raw headlines to backtested signals.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

print('Imports OK')

## Step 1: Load Sample Headlines

In [ ]:
headlines_df = pd.read_csv('../data/sample_headlines.csv')
headlines_df['date'] = pd.to_datetime(headlines_df['date'])
print(f"Loaded {len(headlines_df)} headlines")
headlines_df.head(10)

## Step 2: Sentiment Analysis with Ensemble Model

In [ ]:
from src.models.ensemble import SentimentEnsemble

ensemble = SentimentEnsemble(use_mock_finbert=True)  # Use mock for demo

results = []
for _, row in headlines_df.iterrows():
    r = ensemble.predict(row['headline'])
    results.append({
        'date':           row['date'],
        'ticker':         row['ticker'],
        'headline':       row['headline'],
        'true_label':     row['label'],
        'pred_label':     r.label,
        'ensemble_score': r.ensemble_score,
        'normalized':     r.normalized,
        'confidence':     r.confidence,
    })

results_df = pd.DataFrame(results)
results_df.head(8)

## Step 3: Signal Generation

In [ ]:
from src.signals.signal_generator import SignalGenerator
from src.signals.position_sizing import confidence_based_sizing

gen = SignalGenerator(long_threshold=0.60, short_threshold=0.40, min_confidence=0.55)

signal_rows = []
for _, row in results_df.iterrows():
    size = confidence_based_sizing(row['confidence'], row['normalized'])
    signal = gen.generate(
        ticker=row['ticker'],
        score=row['normalized'],
        confidence=row['confidence'],
        timestamp=row['date'],
        position_size=size,
    )
    signal_rows.append({
        'date':      row['date'],
        'ticker':    row['ticker'],
        'signal':    1 if signal.direction.value == 'LONG' else (-1 if signal.direction.value == 'SHORT' else 0),
        'size':      signal.size,
        'direction': signal.direction.value,
    })

signals_df = pd.DataFrame(signal_rows)

print(f"Total signals: {len(signals_df)}")
print(f"LONG:  {(signals_df['signal'] == 1).sum()}")
print(f"SHORT: {(signals_df['signal'] == -1).sum()}")
print(f"FLAT:  {(signals_df['signal'] == 0).sum()}")

## Step 4: Backtest

In [ ]:
from src.backtest.backtest_engine import VectorizedBacktester, create_sample_backtest
from src.backtest.metrics import compute_metrics, print_metrics

# Use synthetic price data for demo
_, prices_df = create_sample_backtest()

engine = VectorizedBacktester(initial_capital=100_000, commission_bps=5, slippage_bps=3)
result = engine.run(signals_df, prices_df)

print_metrics(result.metrics)

## Step 5: Visualize Results

In [ ]:
from src.backtest.metrics import drawdown_series

fig = plt.figure(figsize=(14, 10))
gs  = gridspec.GridSpec(3, 1, hspace=0.4)

# Equity curve
ax1 = fig.add_subplot(gs[0])
result.equity_curve.plot(ax=ax1, color='#2ecc71', linewidth=2)
ax1.axhline(100_000, color='gray', linestyle='--', alpha=0.5, label='Initial Capital')
ax1.set_title('Portfolio Equity Curve', fontweight='bold')
ax1.set_ylabel('Portfolio Value ($)')
ax1.legend(); ax1.grid(alpha=0.3)

# Drawdown
ax2 = fig.add_subplot(gs[1])
dd  = drawdown_series(result.equity_curve)
dd.plot(ax=ax2, color='#e74c3c', linewidth=1.5)
ax2.fill_between(dd.index, dd.values, 0, alpha=0.3, color='#e74c3c')
ax2.set_title('Drawdown', fontweight='bold')
ax2.set_ylabel('Drawdown (%)')
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax2.grid(alpha=0.3)

# Rolling Sharpe (30-day)
ax3 = fig.add_subplot(gs[2])
rolling_sharpe = result.returns.rolling(63).apply(
    lambda x: x.mean() / x.std() * (252**0.5) if x.std() > 0 else 0
)
rolling_sharpe.plot(ax=ax3, color='#3498db', linewidth=1.5)
ax3.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax3.set_title('Rolling 63-Day Sharpe Ratio', fontweight='bold')
ax3.set_ylabel('Sharpe Ratio')
ax3.grid(alpha=0.3)

plt.savefig('backtest_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved backtest_results.png')

## Step 6: Sentiment Score Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Sentiment score histogram
results_df['ensemble_score'].hist(bins=20, ax=axes[0], color='#3498db', edgecolor='white')
axes[0].set_title('Ensemble Sentiment Score Distribution')
axes[0].set_xlabel('Score (-1=neg, +1=pos)')
axes[0].set_ylabel('Count')
axes[0].grid(alpha=0.3)

# Label breakdown by ticker
ticker_sentiment = results_df.groupby(['ticker', 'pred_label']).size().unstack(fill_value=0)
ticker_sentiment.plot(kind='bar', ax=axes[1], color=['#e74c3c', '#95a5a6', '#2ecc71'])
axes[1].set_title('Sentiment Labels by Ticker')
axes[1].set_xlabel('Ticker')
axes[1].set_ylabel('Count')
axes[1].legend(title='Label')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('sentiment_distribution.png', dpi=150, bbox_inches='tight')
plt.show()